<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/%E5%9D%87%E7%B7%9A%E7%B3%BE%E7%B5%90_LGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [45]:
import warnings
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import yfinance as yf

# 忽略不必要的警告訊息
warnings.filterwarnings("ignore")

# [核心 stock_dict 保持不變]
stock_dict = {
    '0050.TW': '元大台灣50', '0056.TW': '元大高股息', '00878.TW': '國泰永續高股息', '00770.TW': '國泰北美科技',
    '00981A.TW': '統一台股增長主動式', 'SPCX': 'SPACs ETF', 'SOXX': 'iShares半導體ETF', 'SMH': 'VanEck半導體ETF',
    'AAPL': 'Apple 蘋果', 'GOOG': 'Google / Alphabet', 'META': 'Meta', 'MSFT': 'Microsoft 微軟',
    'NVDA': 'NVIDIA 輝達', 'TSM': '台積電 ADR', 'TSLA': 'Tesla 特斯拉', 'ENTG': 'Entegris 英特格',
    'SMR': 'NuScale Power 小型核反應爐', 'BE': 'Bloom Energy 燃料電池', 'JNJ': 'Johnson & Johnson 嬌生',
    'ASML': 'ASML 艾司摩爾', 'AMAT': 'Applied Materials 應用材料', 'LRCX': 'Lam Research 柯林研發',
    'KLAC': 'KLA 科磊', 'AMD': 'AMD 超微', 'AVGO': 'Broadcom 博通', 'QCOM': 'Qualcomm 高通',
    'INTC': 'Intel 英特爾', 'MU': 'Micron 鎂光', 'TXN': 'Texas Instruments 德州儀器', 'ARM': 'ARM 晶心/安謀',
    'MRVL': 'Marvell 邁威爾', 'ADI': 'Analog Devices 亞德諾', 'MPWR': 'Monolithic Power 芯源系統',
    'ON': 'ON Semiconductor 安森美', 'SWKS': 'Skyworks 思佳訊', 'QRVO': 'Qorvo 威訊', 'TER': 'Teradyne 泰瑞達',
    'MKSI': 'MKS Instruments', 'PANW': 'Palo Alto Networks', 'CRWD': 'CrowdStrike', 'FTNT': 'Fortinet',
    'NET': 'Cloudflare', 'ZS': 'Zscaler', 'OKTA': 'Okta', 'S': 'SentinelOne', 'GEN': 'Gen Digital',
    'RPD': 'Rapid7', 'CBRS': 'CyberArk', '2471.TW': '資通', '2480.TW': '敦陽科', '3029.TW': '零壹',
    '6214.TW': '精誠', '3130.TW': '一零四', '2427.TW': '三商電', '3027.TW': '盛達', '5203.TW': '訊連',
    '5471.TW': '松翰', '5410.TWO': '國統', '6183.TW': '關貿', '6203.TWO': '海韻電', '6210.TWO': '慶生',
    '6593.TWO': '台灣銘板', '6689.TW': '伊雲谷', '6690.TWO': '安碁資訊', '6752.TWO': '睿嘉',
    '6763.TWO': '綠界科技', '6865.TWO': '偉康科技', '6874.TWO': '倍力', '6928.TW': '全達',
    '2382.TW': '廣達', '3231.TW': '緯創', '6669.TW': '緯穎', '2317.TW': '鴻海', '2356.TW': '英業達',
    '2324.TW': '仁寶', '2376.TW': '技嘉', '3706.TW': '神達', '2377.TW': '微星', '2357.TW': '華碩',
    '4938.TW': '和碩', '3005.TW': '神基', '2353.TW': '宏碁', '2330.TW': '台積電', '2303.TW': '聯電',
    '2454.TW': '聯發科', '3034.TW': '聯詠', '3661.TW': '世芯-KY', '3443.TW': '創意', '4961.TW': '天鈺',
    '6415.TW': '矽力-KY', '6531.TW': '愛普*', '3035.TW': '智原', '6643.TWO': 'M31', '4966.TWO': '譜瑞-KY',
    '5269.TW': '祥碩', '6104.TWO': '創唯', '6756.TW': '威鋒電子', '2342.TW': '茂矽', '6770.TW': '力積電',
    '3707.TWO': '漢磊', '3016.TW': '嘉晶', '3711.TW': '日月光投控', '2449.TW': '京元電子', '6257.TW': '矽格',
    '3264.TWO': '欣銓', '6239.TW': '力成', '2329.TW': '華泰', '2441.TW': '超豐', '3131.TWO': '弘塑',
    '3583.TW': '辛耘', '6187.TWO': '萬潤', '2467.TW': '志聖', '8027.TWO': '钛昇', '5434.TW': '崇越',
    '3010.TW': '華立', '1560.TW': '中砂', '3680.TWO': '家登', '5234.TW': '達興材料', '4749.TWO': '新應材',
    '8028.TW': '昇陽半導體', '6515.TW': '穎崴', '6683.TWO': '雍智科技', '6510.TWO': '精測', '6223.TWO': '旺矽',
    '2404.TW': '漢唐', '1773.TW': '勝一', '6196.TW': '帆宣', '6139.TW': '亞翔', '6613.TWO': '朋億*',
    '4755.TW': '三福化', '4768.TWO': '晶呈科技', '3563.TW': '牧德', '3167.TW': '大量', '6438.TW': '迅得',
    '1595.TWO': '川寶', '6147.TWO': '頎邦', '8150.TW': '南茂', '6552.TW': '易華電', '5536.TWO': '聖暉*',
    '3644.TWO': '凌嘉科', '7769.TW': '鴻勁', '2344.TW': '華邦電', '2408.TW': '南亞科', '2337.TW': '旺宏',
    '3006.TW': '晶豪科', '3260.TWO': '威剛', '2451.TW': '創見', '4967.TW': '十銓', '8271.TW': '宇瞻',
    '5289.TWO': '宜晶', '8299.TWO': '群聯', '5351.TWO': '鈺創', '2308.TW': '台達電', '2301.TW': '光寶科',
    '6282.TW': '康舒', '6412.TW': '群電', '3665.TW': '貿聯-KY', '3017.TW': '奇鋐', '3324.TWO': '雙鴻',
    '3653.TW': '健策', '2421.TW': '建準', '8996.TW': '高力', '3483.TWO': '力致', '6230.TW': '尼得科超眾',
    '3013.TW': '晟銘電', '6805.TW': '富世達', '8210.TW': '勤誠', '6117.TW': '迎廣', '6235.TW': '華孚',
    '2354.TW': '鴻準', '3376.TW': '新日興', '3548.TWO': '兆利', '5243.TW': '乙盛-KY', '6715.TW': '嘉基',
    '3533.TW': '嘉澤', '3217.TWO': '優群', '3023.TW': '信邦', '2392.TW': '正崴', '3689.TWO': '湧德',
    '3357.TWO': '臺慶科', '6862.TW': '三集瑞-KY', '6821.TWO': '聯寶', '3207.TWO': '耀勝', '6197.TW': '佳必琪',
    '8103.TW': '瀚荃', '3526.TWO': '凡甲', '3605.TW': '宏致', '2059.TW': '川湖', '6584.TWO': '南俊國際',
    '2327.TW': '國巨', '2492.TW': '華新科', '2375.TW': '凱美', '2478.TW': '大毅', '3026.TW': '禾伸堂',
    '3090.TW': '日電貿', '6173.TWO': '信昌電', '6155.TW': '鈞寶', '6175.TWO': '立敦', '5328.TWO': '華容',
    '3236.TWO': '千如', '8043.TWO': '蜜望實', '3037.TW': '欣興', '8046.TW': '南電', '3189.TW': '景碩',
    '4958.TW': '臻鼎-KY', '2368.TW': '金像電', '3044.TW': '健鼎', '2313.TW': '華通', '8155.TWO': '博智',
    '2383.TW': '台光電', '6274.TWO': '台燿', '6213.TW': '聯茂', '1717.TW': '長興', '1815.TWO': '富喬',
    '1802.TW': '台玻', '5340.TWO': '建榮', '5475.TWO': '德宏', '3305.TW': '昇貿', '3631.TWO': '晟楠',
    '8358.TWO': '金居', '8021.TW': '尖點', '6672.TW': '騰輝電子-KY', '2345.TW': '智邦', '5388.TW': '中磊',
    '3558.TWO': '神準', '3704.TW': '合勤控', '4906.TW': '正文', '4979.TWO': '華星光', '6442.TW': '光聖',
    '4908.TWO': '前鼎', '3163.TWO': '波若威', '3450.TW': '聯鈞', '6426.TW': '統新', '4977.TW': '眾達-KY',
    '6530.TWO': '創威', '3363.TWO': '上詮', '3234.TWO': '光環', '4903.TWO': '聯光通', '3081.TWO': '聯亞',
    '4991.TWO': '環宇-KY', '4971.TWO': 'IET-KY', '6588.TWO': '東典光電', '3491.TWO': '昇達科', '2314.TW': '台揚',
    '6285.TW': '啟碁', '3105.TWO': '穩懋', '2455.TW': '全新', '3138.TW': '耀登', '2419.TW': '仲琦',
    '2395.TW': '研華', '6166.TW': '凌華', '8050.TWO': '廣積', '3556.TWO': '禾瑞亞', '2414.TW': '精技',
    '6414.TW': '樺漢', '3022.TW': '威強電', '2397.TW': '友通', '5314.TWO': '世紀', '6781.TW': 'AES-KY',
    '3211.TWO': '順達', '6121.TWO': '新普', '3323.TWO': '加百裕', '3625.TWO': '西勝', '8038.TWO': '長園科',
    '4931.TWO': '新盛力', '1519.TW': '華城', '1513.TW': '中興電', '1514.TW': '亞力', '1503.TW': '士電',
    '1609.TW': '大亞', '1605.TW': '華新', '1608.TW': '華榮', '6869.TW': '雲豹能源', '2049.TW': '上銀',
    '4576.TW': '大銀微系統', '4585.TW': '達明', '2359.TW': '所羅門', '6188.TWO': '廣明', '8374.TW': '羅昇',
    '5443.TWO': '均豪', '6640.TWO': '均華', '2464.TW': '盟立', '6215.TW': '和椿', '4562.TW': '穎漢',
    '1590.TW': '亞德客-KY', '1504.TW': '東元', '3481.TW': '群創', '2409.TW': '友達', '3008.TW': '大立光',
    '4915.TW': '先進光', '5288.TW': '匯鑽科', '2393.TW': '億光', '2201.TW': '裕隆', '2204.TW': '中華',
    '2206.TW': '三陽工業', '1536.TW': '和大', '2231.TW': '聯嘉', '3552.TWO': '同致', '6279.TWO': '胡連',
    '2603.TW': '長榮', '2609.TW': '陽明', '2615.TW': '萬海', '2605.TW': '新興', '2606.TW': '裕民',
    '2612.TW': '中航', '2617.TW': '台航', '2637.TW': '慧洋-KY', '2641.TWO': '正德', '5608.TW': '四維航',
    '2610.TW': '華航', '2618.TW': '長榮航', '2630.TW': '亞航', '5603.TWO': '陸海', '2607.TW': '榮運',
    '2608.TW': '嘉里大榮', '2611.TW': '志信', '2613.TW': '中櫃', '2636.TW': '台驊投控', '2642.TW': '宅配通',
    '2633.TW': '台灣高鐵', '5607.TW': '遠雄港', '5609.TWO': '中菲行', '8367.TW': '建新國際', '2892.TW': '第一金',
    '5880.TW': '合庫金', '1210.TW': '大成', '1215.TW': '卜蜂', '1216.TW': '統一', '2912.TW': '統一超',
    '5903.TWO': '全家', '1303.TW': '南亞', '2465.TW': '麗臺', '8163.TW': '達方', '3042.TW': '晶技',
    '8182.TWO': '加高', '3229.TW': '泰藝', '3308.TW': '聯傑', '6284.TWO': '佳邦', '2484.TW': '希華', '8088.TWO': '華信科'
}

# --- 核心優化參數與流程 ---
model_params = {
    'n_estimators': 300, 'learning_rate': 0.03, 'num_leaves': 15,
    'feature_fraction': 0.8, 'bagging_fraction': 0.7,
    'lambda_l1': 0.1, 'lambda_l2': 0.1, 'scale_pos_weight': 2.0,
    'random_state': 42, 'verbose': -1
}

WIN_PROB_THRESHOLD = 0.45

market_df = yf.download("^TWII", period="3y", progress=False)
if isinstance(market_df.columns, pd.MultiIndex): market_df.columns = market_df.columns.get_level_values(0)

all_tickers = list(stock_dict.keys())
batch_data = yf.download(all_tickers, period="3y", progress=False, group_by="ticker")

predictions = []
for ticker, name in stock_dict.items():
    try:
        df = batch_data[ticker].dropna(how='all')
        if len(df) < 200: continue
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

        # 篩選條件
        ma5, ma10, ma20 = df['Close'].rolling(5).mean(), df['Close'].rolling(10).mean(), df['Close'].rolling(20).mean()
        ma_tangle = (pd.concat([ma5, ma10, ma20], axis=1).max(axis=1) - pd.concat([ma5, ma10, ma20], axis=1).min(axis=1)) / df['Close'] <= 0.03
        vol_ok = df['Volume'].rolling(5).mean() >= 500000

        # 標籤定義
        horizon = 10
        f_high = df['High'].shift(-horizon).rolling(window=horizon).max().shift(horizon-1)
        df['Target'] = ((f_high - df['Close']) / df['Close'] >= 0.09).astype(int)

        df_feat, f_cols = compute_features(df, market_df)
        df_clean = df_feat.dropna(subset=f_cols + ['Target'])

        dt = df_clean.index[-1]
        if not (ma_tangle.loc[dt] and vol_ok.loc[dt]): continue

        train_size = int(len(df_clean) * 0.8)
        train_df = df_clean.iloc[:train_size]
        if len(train_df) < 50: continue

        clf = lgb.LGBMClassifier(**model_params)
        clf.fit(train_df[f_cols], train_df['Target'])

        prob = clf.predict_proba(df_clean.loc[[dt], f_cols])[0][1]
        predictions.append({
            "預測日期": dt.strftime('%Y-%m-%d'), "股票名稱": name, "股票代號": ticker, "raw_prob": prob, "成功機率": f"{prob*100:.2f}%"
        })
    except: continue

final_output_df = pd.DataFrame(predictions)
if not final_output_df.empty:
    final_output_df = final_output_df.sort_values(by="raw_prob", ascending=False)
    high_win_df = final_output_df[final_output_df['raw_prob'] >= WIN_PROB_THRESHOLD]

    print(f"\n=== [優化版] 高勝率訊號清單 (門檻 > {WIN_PROB_THRESHOLD*100:.0f}%) ===")
    display(high_win_df[["預測日期", "股票名稱", "成功機率"]])

    # 整合：前五大標的指標對比表
    print("\n=== 前五大標的技術指標與市場強度對比 ===")
    top_5 = high_win_df.head(5)
    comp_results = []
    for _, row in top_5.iterrows():
        t = row['股票代號']
        df_t = batch_data[t].dropna(how='all')
        if isinstance(df_t.columns, pd.MultiIndex): df_t.columns = df_t.columns.get_level_values(0)
        df_feat_t, _ = compute_features(df_t, market_df)
        latest = df_feat_t.iloc[-1]
        comp_results.append({
            '股票名稱': row['股票名稱'],
            'Alpha_5d': f"{latest['Alpha_5d']*100:.2f}%",
            '價格斜率': round(latest['Close_Slope'], 3),
            '量能噴發': round(latest['Volume_Explosion'], 2),
            '換手強度': round(latest['Turnover_Rate'], 2),
            '五日乖離': f"{latest['BIAS_5']*100:.2f}%"
        })
    display(pd.DataFrame(comp_results))
else:
    print("今日無符合標的。")


=== [優化版] 高勝率訊號清單 (門檻 > 45%) ===


,預測日期,股票名稱,成功機率
63,2026-09-02,宇瞻,99.26%
121,2026-09-02,加高,97.78%
11,2026-09-01,NuScale Power 小型核反應爐,97.46%
59,2026-09-02,旺宏,97.32%
14,2026-09-01,Lam Research 柯林研發,96.86%
56,2026-09-02,亞翔,96.30%
83,2026-09-02,景碩,95.27%
82,2026-09-02,蜜望實,94.17%
28,2026-09-02,仁寶,93.28%
58,2026-09-02,華邦電,92.43%



=== 前五大標的技術指標與市場強度對比 ===


,股票名稱,Alpha_5d,價格斜率,量能噴發,換手強度,五日乖離
0,宇瞻,-6.62%,-1.650,0.71,0.44,-1.82%
1,加高,-6.70%,-0.130,0.78,0.11,-0.17%
2,NuScale Power 小型核反應爐,0.43%,-0.106,0.56,0.71,-2.17%
3,旺宏,-4.47%,-0.750,0.97,0.42,-2.47%
4,Lam Research 柯林研發,-5.98%,-1.344,1.11,0.60,-3.23%


In [41]:
import pandas as pd
from google.colab import files

# 1. 取得最新預測日期中機率最高的前五大標的
latest_date = final_output_df['預測日期'].max()
top_5_stocks = final_output_df[final_output_df['預測日期'] == latest_date].head(5)
top_5_list = top_5_stocks['股票名稱'].tolist()

print(f"--- 針對最新日期 ({latest_date}) 前五大標的進行歷史勝率檢視 ---")

# 2. 從歷史明細中提取這些標的的表現
analysis_results = []
for name in top_5_list:
    stock_data = hist_detail_df[hist_detail_df['股票名稱'] == name]
    if not stock_data.empty:
        total = len(stock_data)
        success = len(stock_data[stock_data['驗證結果'] == '成功'])
        win_rate = (success / total) * 100
        avg_ret = stock_data['期間最高漲幅'].str.replace('%', '').astype(float).mean()

        analysis_results.append({
            '股票名稱': name,
            '歷史總訊號數': total,
            '成功次數': success,
            '歷史實測勝率': f"{win_rate:.2f}%",
            '歷史平均最高漲幅': f"{avg_ret:.2f}%"
        })
    else:
        analysis_results.append({
            '股票名稱': name,
            '歷史總訊號數': 0,
            '成功次數': 0,
            '歷史實測勝率': "無歷史數據",
            '歷史平均最高漲幅': "N/A"
        })

# 3. 建立 DataFrame 並顯示
top_5_analysis_df = pd.DataFrame(analysis_results)
display(top_5_analysis_df)

# 4. 匯出 Excel 並下載
file_name = 'top_5_historical_win_rate.xlsx'
top_5_analysis_df.to_excel(file_name, index=False)
print(f"\n檔案 {file_name} 已產生，正在啟動自動下載...")
files.download(file_name)

--- 針對最新日期 (2026-09-02) 前五大標的進行歷史勝率檢視 ---


,股票名稱,歷史總訊號數,成功次數,歷史實測勝率,歷史平均最高漲幅
0,宇瞻,3,3,100.00%,73.48%
1,旺宏,3,3,100.00%,42.10%
2,加高,3,1,33.33%,12.71%
3,景碩,3,3,100.00%,20.67%
4,仁寶,3,0,0.00%,1.29%



檔案 top_5_historical_win_rate.xlsx 已產生，正在啟動自動下載...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
import pandas as pd

# 1. 定義欲分析的標的列表
target_names = ['力積電', '景碩', '新盛力', '旺宏', '力成']
target_tickers = ['6770.TW', '3189.TW', '4931.TWO', '2337.TW', '6239.TW']

comparison_results = []

for ticker, name in zip(target_tickers, target_names):
    try:
        # 獲取單一股票數據並計算特徵
        df = batch_data[ticker].dropna(how='all')
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

        df_feat, f_cols = compute_features(df, market_df)
        latest_feat = df_feat.iloc[-1] # 取最新交易日數據

        comparison_results.append({
            '股票名稱': name,
            'Alpha_5d (市場強度)': f"{latest_feat['Alpha_5d']*100:.2f}%",
            'Close_Slope (價格斜率)': round(latest_feat['Close_Slope'], 3),
            'Volume_Explosion (量能噴發)': round(latest_feat['Volume_Explosion'], 2),
            'Turnover_Rate (換手強度)': round(latest_feat['Turnover_Rate'], 2),
            'BIAS_5 (五日乖離)': f"{latest_feat['BIAS_5']*100:.2f}%",
            'NATR (波動率)': f"{latest_feat['NATR']*100:.2f}%"
        })
    except Exception as e:
        print(f"無法處理 {name}: {e}")

# 2. 建立對比表
comparison_df = pd.DataFrame(comparison_results)
print("=== 前五大標的技術指標與市場強度對比表 ===")
display(comparison_df)

=== 前五大標的技術指標與市場強度對比表 ===


,股票名稱,Alpha_5d (市場強度),Close_Slope (價格斜率),Volume_Explosion (量能噴發),Turnover_Rate (換手強度),BIAS_5 (五日乖離),NATR (波動率)
0,力積電,-1.94%,0.506,1.06,0.74,0.18%,5.72%
1,景碩,-0.21%,-10.900,1.25,2.00,-0.39%,7.00%
2,新盛力,1.44%,0.350,0.49,0.75,-0.80%,6.11%
3,旺宏,-4.47%,-0.750,0.97,0.42,-2.47%,5.01%
4,力成,-1.05%,-0.400,0.74,0.62,0.64%,4.65%
